# Connecting to Azure SQL Database (AdventureWorks)

This notebook connects to your Azure SQL Database:
- **Server:** `dp900-sql-server-teslim.database.windows.net`
- **Database:** `AdventureWorks`

## Prerequisites ✅
- ODBC Driver 18 for SQL Server (Already installed!)
- Python packages: `pyodbc`, `pandas` (Already installed!)

## Verify ODBC Driver Installation

In [1]:
import pyodbc

# List all available ODBC drivers
drivers = [driver for driver in pyodbc.drivers()]
print("Available ODBC drivers:")
for driver in drivers:
    print(f"  ✓ {driver}")

# Check for SQL Server driver
sql_drivers = [d for d in drivers if 'SQL Server' in d]
if sql_drivers:
    print(f"\n✓ SQL Server drivers found: {sql_drivers[0]}")
else:
    print("\n❌ No SQL Server driver found!")

Available ODBC drivers:
  ✓ ODBC Driver 18 for SQL Server

✓ SQL Server drivers found: ODBC Driver 18 for SQL Server


## Connect to Azure SQL Database - AdventureWorks

This will prompt you to log in via your browser (Azure AD Interactive Authentication).

In [2]:
import pyodbc
import pandas as pd

# Your Azure SQL Database connection details
SERVER = 'dp900-sql-server-teslim.database.windows.net'
DATABASE = 'AdventureWorks'

# Connection string with Azure AD Interactive authentication
conn_string = f"""
    Driver={{ODBC Driver 18 for SQL Server}};
    Server={SERVER};
    Database={DATABASE};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
"""

try:
    # Connect to the database
    print("🔄 Connecting to Azure SQL Database...")
    print("   (A browser window will open for authentication)")
    conn = pyodbc.connect(conn_string)
    print("\n✅ Connection successful!")
    print(f"   Connected to: {DATABASE} on {SERVER}")
    
    # Store connection for later use
    print("\n✓ Connection ready for queries")
    
except pyodbc.Error as e:
    print(f"\n❌ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure you have access to the Azure SQL Database")
    print("2. Check your Azure AD credentials")
    print("3. Verify the server and database names are correct")
    print("4. Ensure your IP is allowed in Azure SQL firewall rules")

🔄 Connecting to Azure SQL Database...
   (A browser window will open for authentication)

❌ Connection failed: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')

Troubleshooting:
1. Make sure you have access to the Azure SQL Database
2. Check your Azure AD credentials
3. Verify the server and database names are correct
4. Ensure your IP is allowed in Azure SQL firewall rules


## List All Tables in AdventureWorks Database

In [ ]:
# Query to list all tables
query_tables = """
SELECT 
    TABLE_SCHEMA as [Schema],
    TABLE_NAME as [Table],
    TABLE_TYPE as [Type]
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_SCHEMA, TABLE_NAME
"""

try:
    df_tables = pd.read_sql(query_tables, conn)
    print(f"\n📊 Found {len(df_tables)} tables in AdventureWorks database:\n")
    print(df_tables.to_string(index=False))
    
except Exception as e:
    print(f"Error listing tables: {e}")

## Example Queries - Explore AdventureWorks Data

In [ ]:
# Example 1: Query the first table from the list above
# Replace 'YourSchema.YourTable' with an actual table name from the list above

query = """
SELECT TOP 10 *
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'YourTableName'
ORDER BY ORDINAL_POSITION
"""

# Uncomment and run after replacing with actual table name:
# df = pd.read_sql(query, conn)
# print(df)

## Query Sales Data (Common AdventureWorks Tables)

In [ ]:
# Try querying common AdventureWorks tables
# (These table names may vary based on your AdventureWorks version)

# Example: Get product information
query_products = """
SELECT TOP 10
    ProductID,
    Name,
    ProductNumber,
    Color,
    StandardCost,
    ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC
"""

try:
    df_products = pd.read_sql(query_products, conn)
    print("\n🛍️ Top 10 Most Expensive Products:\n")
    print(df_products.to_string(index=False))
except Exception as e:
    print(f"Note: Table might not exist in your database. Error: {e}")
    print("Check the table list above for actual table names.")

## Query Customer Data

In [ ]:
# Example: Get customer information
query_customers = """
SELECT TOP 10
    CustomerID,
    FirstName,
    LastName,
    CompanyName,
    EmailAddress
FROM SalesLT.Customer
ORDER BY CustomerID
"""

try:
    df_customers = pd.read_sql(query_customers, conn)
    print("\n👥 First 10 Customers:\n")
    print(df_customers.to_string(index=False))
except Exception as e:
    print(f"Note: Table might not exist. Error: {e}")
    print("Use the table names from the list above.")

## Custom Query - Write Your Own SQL

In [ ]:
# Write your custom SQL query here
custom_query = """
-- Replace this with your SQL query
SELECT 
    'Hello from Azure SQL!' as Message,
    GETDATE() as CurrentDateTime
"""

try:
    df_custom = pd.read_sql(custom_query, conn)
    print("\n📊 Query Results:\n")
    print(df_custom.to_string(index=False))
except Exception as e:
    print(f"Query error: {e}")

## Reusable Connection Helper Class

In [ ]:
class AzureSQLConnection:
    """Helper class for Azure SQL Database connections."""
    
    def __init__(self, server: str, database: str):
        self.server = server
        self.database = database
        self.conn = None
    
    def connect(self) -> pyodbc.Connection:
        """Establish connection to Azure SQL Database."""
        conn_string = f"""
            Driver={{ODBC Driver 18 for SQL Server}};
            Server={self.server};
            Database={self.database};
            Authentication=ActiveDirectoryInteractive;
            Encrypt=yes;
            TrustServerCertificate=no;
        """
        
        try:
            self.conn = pyodbc.connect(conn_string)
            print(f"✓ Connected to {self.database}")
            return self.conn
        except pyodbc.Error as e:
            raise ConnectionError(f"Failed to connect: {e}")
    
    def query(self, sql: str) -> pd.DataFrame:
        """Execute SQL query and return results as DataFrame."""
        if not self.conn:
            self.connect()
        return pd.read_sql(sql, self.conn)
    
    def list_tables(self, schema: str = None) -> pd.DataFrame:
        """List all tables in the database."""
        where_clause = f"WHERE TABLE_SCHEMA = '{schema}'" if schema else ""
        query = f"""
        SELECT 
            TABLE_SCHEMA as [Schema],
            TABLE_NAME as [Table],
            TABLE_TYPE as [Type]
        FROM INFORMATION_SCHEMA.TABLES
        {where_clause}
        ORDER BY TABLE_SCHEMA, TABLE_NAME
        """
        return self.query(query)
    
    def get_table_info(self, table_name: str, schema: str = None) -> pd.DataFrame:
        """Get column information for a specific table."""
        where_clause = f"AND TABLE_SCHEMA = '{schema}'" if schema else ""
        query = f"""
        SELECT 
            COLUMN_NAME as [Column],
            DATA_TYPE as [Type],
            CHARACTER_MAXIMUM_LENGTH as [MaxLength],
            IS_NULLABLE as [Nullable]
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = '{table_name}' {where_clause}
        ORDER BY ORDINAL_POSITION
        """
        return self.query(query)
    
    def close(self):
        """Close the connection."""
        if self.conn:
            self.conn.close()
            print("✓ Connection closed")


# Example usage of the helper class
print("\n📦 AzureSQLConnection class ready to use!")
print("\nExample usage:")
print("  db = AzureSQLConnection('dp900-sql-server-teslim.database.windows.net', 'AdventureWorks')")
print("  db.connect()")
print("  df = db.query('SELECT TOP 10 * FROM YourTable')")
print("  db.close()")

## Close Connection When Done

In [ ]:
# Always close the connection when you're done
try:
    conn.close()
    print("✓ Connection closed successfully")
except:
    print("Connection was already closed or not established")

## Troubleshooting Common Issues

### 1. Firewall Issues
If you get a connection timeout, your IP address might not be allowed:
- Go to Azure Portal → SQL Server → Firewalls and virtual networks
- Add your current IP address to the allowed list

### 2. Authentication Failed
- Make sure you're using the correct Azure AD account
- Check that your account has access to the database
- Try using SQL Authentication instead:
  ```python
  conn_string = f"""
      Driver={{ODBC Driver 18 for SQL Server}};
      Server={SERVER};
      Database={DATABASE};
      UID=your_username;
      PWD=your_password;
      Encrypt=yes;
  """
  ```

### 3. Table Not Found
- Run the "List All Tables" cell first to see actual table names
- AdventureWorks tables are usually in schemas like: `SalesLT`, `dbo`, `Production`, `Sales`
- Use fully qualified names: `SchemaName.TableName`